# Agentic Action With Retrieval

## Background

The OCR notebook showed a bad extraction silently propagate into a wrong *answer*. The retrieval notebook showed the same failure one layer earlier, a bad search result silently propagating into a wrong *answer*.

This notebook goes one step further: a bad search result propagating into a wrong *action*. The agent here doesn't just answer a question, it decides whether to approve a customer refund and actually calls a tool to do it. If retrieval hands it a superseded policy, it doesn't just say something wrong, it takes an action it shouldn't have taken, with the same full confidence either way.

This is the sharper version of the talk's point: when an agent only answers, a bad retrieval produces a wrong sentence someone can catch and correct. When an agent acts, a bad retrieval produces a wrong *outcome*, a refund approved that shouldn't have been, before anyone gets a chance to check.

### The Scenario

A customer requests a refund \$75. 
CloudDeploy's **current** policy : refunds under <span>$</span>50 can be auto-approved by a support agent, \$50 or more needs manager review. 
There's also a **superseded** internal handbook still sitting in the knowledge base, written before a policy tightening, that says refunds under \$100 can be auto-approved.

The two policy documents are written almost identically, same structure, same key terms, only the dollar figure differs. That's deliberate and realistic: policy updates are usually small, focused edits, not full rewrites, so the old and new versions of a policy doc are often nearly indistinguishable by keyword. A literal-match search genuinely cannot tell them apart. This isn't a weakened tool, it's an accurate simulation of a real, common knowledge-base problem: stale duplicates of policy content that read almost exactly like the current version.

## Import Libraries

In [48]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [49]:
!pip freeze | grep qdrant-client

python3.12(57118) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


qdrant-client==1.18.0


In [50]:
!pip freeze | grep fastembed

python3.12(59662) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


fastembed==0.6.0


In [51]:
!pip freeze | grep langchain

python3.12(59763) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


langchain==1.2.0
langchain-classic==1.0.1
langchain-community==0.3.14
langchain-core==1.2.5
langchain-ollama==1.0.1
langchain-openai==1.1.6
langchain-protocol==0.0.18
langchain-text-splitters==0.3.5


In [52]:
!pip install rank_bm25

python3.12(59768) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.



[notice] A new release of pip is available: 25.3 -> 26.2
[notice] To update, run: pip install --upgrade pip


In [53]:
from qdrant_client import QdrantClient
from qdrant_client.models import PointStruct, VectorParams, Distance, Filter, FieldCondition, MatchValue
from fastembed import TextEmbedding
import re
from collections import Counter
from langchain.tools import tool
from langchain_community.retrievers import BM25Retriever

### Build the Knowledge Base

Note the two refund policy documents are ordered with the superseded one first. That's not an artificial trick, it mirrors a very ordinary real-world situation: older content is often indexed earlier, accumulates more inbound links or crawl history, or simply sits higher in a legacy system's default sort order, and without an explicit freshness signal, there's no reason a search index would rank the newer one first.

In [54]:
corpus = [
    {
        "id": 1,
        "title": "API Rate Limits",
        "status": "current",
        "text": (
            "CloudDeploy API rate limits by plan: Free tier is capped at 100 requests "
            "per minute. Pro tier is capped at 1,000 requests per minute. Enterprise tier "
            "is capped at 10,000 requests per minute, negotiable up to 50,000 with a "
            "custom agreement. These limits were last updated in the current release."
        ),
    },
    {
        "id": 2,
        "title": "Changelog v1.2",
        "status": "deprecated",
        "text": (
            "Changelog v1.2: As of this release, Pro tier requests are capped at 500 "
            "requests per minute. Free tier remains capped at 100 requests per minute. "
            "This changelog entry describes limits from an earlier release and has since "
            "been superseded."
        ),
    },
    {
        "id": 3,
        "title": "Onboarding Guide",
        "status": "current",
        "text": (
            "To get started with CloudDeploy, create an account, generate an API key from "
            "the dashboard, and install the CLI. New accounts start on the Free tier by "
            "default and can upgrade to Pro or Enterprise at any time from billing settings."
        ),
    },
    {
        "id": 4,
        "title": "Support FAQ",
        "status": "current",
        "text": (
            "Common support questions: password resets are handled via the account "
            "settings page. API keys can be rotated at any time without downtime. Billing "
            "is monthly by default, with annual billing available at a discount."
        ),
    },
    {
        "id": 5,
        "title": "Security Overview",
        "status": "superseded",
        "text": (
            "CloudDeploy encrypts data at rest and in transit. No logging for the requests is available."
        ),
    },
    {
        "id": 6,
        "title": "Account Termination",
        "status": "current",
        "text": (
        "To terminate your CloudDeploy account, submit a request through the billing "
        "portal. Termination takes effect at the end of the current billing cycle. "
        "All API keys are revoked immediately upon account termination, and stored "
        "project data is deleted after 30 days."
        ),
    },
    {
        "id": 7,
        "title": "Refund Policy",
        "status": "superseded",
        "text": (
            "Refund policy: refunds under 100 dollars can be auto approved by "
            "support agents without manager review. Refunds of 100 dollars or "
            "more require manager approval before processing. For example, a "
            "customer refund request of 80 dollars would be auto approved by a "
            "support agent under this refund policy without further review."
        ),
    },
    {
        "id": 8,
        "title": "Refund Policy",
        "status": "current",
        "text": (
            "Refund policy: refunds under 50 dollars can be auto approved by "
            "support agents without manager review. Refunds of 50 dollars or "
            "more require manager approval before processing."
        ),
    },
    {
        "id": 9,
        "title": "Security Overview",
        "status": "current",
        "text": (
            "CloudDeploy encrypts data at rest and in transit. API keys are scoped per "
            "project and can be restricted by IP allowlist. All requests are logged for "
            "30 days for audit purposes."
        ),
    },
    {
        "id": 10,
        "title": "Changelog v1.3",
        "status": "current",
        "text": (
            "Changelog v1.3: As of this release, Pro tier requests are capped at 1000 "
            "requests per minute. Free tier remains capped at 100 requests per minute. "
        ),
    },
]


### Create the Retrieval Tool (V1 - Naive)

The naive tool scores documents purely on literal keyword overlap, no semantic understanding, and critically, no notion of which document is current versus deprecated. A changelog entry that happens to share more literal wording with the query can easily outscore the actual current doc.

In [55]:
STOPWORDS = {
    "a", "an", "the", "is", "are", "of", "for", "to", "in", "on", "at",
    "and", "or", "what", "this", "that", "as", "s", "how", "do", "i", "my",
    "can", "get", "does", "it", "be", "have", "has", "you", "your", "if",
    "will", "with", "from", "by", "should",
}

def _preprocess(text: str) -> list[str]:
    words = re.findall(r"[a-z0-9']+", text.lower())
    return [w for w in words if w not in STOPWORDS]

# converting docs to langchain-friendly format
texts = [doc["text"] for doc in corpus]
metadatas = [{"title": doc["title"], "status": doc["status"]} for doc in corpus]

In [56]:
naive_retriever = BM25Retriever.from_texts(
    texts, metadatas=metadatas, preprocess_func=_preprocess, k=1
)

@tool
def naive_search(query: str) -> str:
    """BM25 keyword search over the CloudDeploy knowledge base. Returns the single best-matching document by lexical relevance."""
    result = naive_retriever.invoke(query)[0]
    return f"[{result.metadata['title']}]\n{result.page_content}"

#### Run it on the question of interest regarding refunds

In [57]:
refund_question = "A customer is requesting a 75 dollar refund. Should this be auto approved?"

In [58]:
naive_search_result = naive_search.run(refund_question)
print("Naive Retrieval Output:\n------------------------\n", naive_search_result)


Naive Retrieval Output:
------------------------
 [Refund Policy]
Refund policy: refunds under 100 dollars can be auto approved by support agents without manager review. Refunds of 100 dollars or more require manager approval before processing. For example, a customer refund request of 80 dollars would be auto approved by a support agent under this refund policy without further review.


#### Note:
Both refund policy documents score identically, the query has no way to distinguish "under 100 dollars" from "under 50 dollars" by keyword overlap alone, both contain the same surrounding words. With no tiebreaker, the tool returns whichever comes first, here, the superseded $100 threshold. That's the whole failure mode in one cell: not "no match found," but a confidently returned, plausible-looking, wrong policy.

### A note on filtering

It's fair to ask: wouldn't adding a `status == "current"` filter to `naive_search` have fixed this too? 
Yes, it would have. 
Metadata filtering isn't a semantic-search feature, it works identically in front of BM25 or embeddings. 
Once documents are tagged with a freshness signal and a filter is wired in, either ranking approach returns the correct policy.

So this demo isn't proof that semantic search is inherently better at freshness, it's proof that *neither* approach gets freshness for free.
Someone has to tag the data and add the filter, regardless of which retrieval method sits underneath.

Where semantic search earns its keep with no filter required is a different failure mode entirely: 
vocabulary mismatch (e.g. a user asking to "cancel my subscription" when the docs only say "terminate your account"). 
No metadata filter fixes that, there's no shared token for a filter or a keyword match to grab onto. 
That's the case where embeddings are doing genuinely irreplaceable work, understanding meaning, 
not just resolving which of two similar documents is newer.

In [79]:
another_question = "How can one cancel their contract?"

In [80]:
another_search_result = naive_search.run(another_question)
print("Naive Retrieval Output for another question:\n------------------------\n", another_search_result)

Naive Retrieval Output for another question:
------------------------
 [Changelog v1.3]
Changelog v1.3: As of this release, Pro tier requests are capped at 1000 requests per minute. Free tier remains capped at 100 requests per minute. 


### Create a better Retrieval Tool with Semantic understanding and filters

The semantic version embeds the corpus with FastEmbed into an in-memory Qdrant collection, and explicitly filters on the `status` field. This is the realistic version of "strong retrieval": embeddings alone won't reliably infer which of two near-identical policy documents is current, that's not a semantic property, so a production system adds a metadata/freshness signal on top. That combination, meaning-aware search plus an explicit currency check, is what actually closes this gap, not one or the other alone.

In [61]:
embedding_model = TextEmbedding(model_name="BAAI/bge-small-en-v1.5")

client = QdrantClient(":memory:")
client.create_collection(
    collection_name="clouddeploy_policies",
    vectors_config=VectorParams(size=384, distance=Distance.COSINE),
)

titles = [doc["title"] for doc in corpus]
texts = [doc["text"] for doc in corpus]
statuses = [doc["status"] for doc in corpus]
vectors = list(embedding_model.embed(texts))

points = [
    PointStruct(
        id=i,
        vector=vector.tolist(),
        payload={"title": title, "text": text, "status": status},
    )
    for i, (title, text, status, vector) in enumerate(zip(titles, texts, statuses, vectors))
]
client.upsert(collection_name="clouddeploy_policies", points=points)


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/706 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model_optimized.onnx:   0%|          | 0.00/66.5M [00:00<?, ?B/s]

UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

In [62]:
@tool
def semantic_search_with_filter(query: str) -> str:
    """Semantic search over the CloudDeploy policy knowledge base, restricted to current documents only. Returns the single best-matching current document by meaning."""
    query_vector = list(embedding_model.embed([query]))[0].tolist()

    results = client.query_points(
        collection_name="clouddeploy_policies",
        query=query_vector,
        query_filter=Filter(
            must=[FieldCondition(key="status", match=MatchValue(value="current"))]
        ),
        limit=1,
    ).points

    best = results[0].payload
    return f"[{best['title']}]\n{best['text']}"


Run it on the same question.

In [70]:
semantic_search_result = semantic_search_with_filter.run(refund_question)
print("Semantic Retrieval Output:\n--------------------------\n", semantic_search_result)


Semantic Retrieval Output:
--------------------------
 [Refund Policy]
Refund policy: refunds under 50 dollars can be auto approved by support agents without manager review. Refunds of 50 dollars or more require manager approval before processing.


In [81]:
## or like "What is the latest info you have on refunds"
semantic_search_result_another_question = semantic_search_with_filter.run(another_question)
print("Semantic Retrieval Output for another question:\n--------------------------\n", semantic_search_result_another_question)


Semantic Retrieval Output for another question:
--------------------------
 [Account Termination]
To terminate your CloudDeploy account, submit a request through the billing portal. Termination takes effect at the end of the current billing cycle. All API keys are revoked immediately upon account termination, and stored project data is deleted after 30 days.


### Give the Agent an Action, Not Just an Answer

Two tools this time: the search tool, and an action tool the agent actually calls to do something. `approve_refund` and `escalate_for_review` are stand-ins for a real billing system call, print statements only, but the point is the agent has to pick one and commit, exactly like a real agent hitting a real API would.

In [76]:
@tool
def approve_refund(amount: float) -> str:
    """Approve and process a customer refund for the given dollar amount without manager review."""
    return f"ACTION TAKEN: Refund of ${amount:.2f} auto-approved and processed. No manager review requested."

@tool
def escalate_for_review(amount: float) -> str:
    """Escalate a customer refund request for manager review instead of approving it directly."""
    return f"ACTION TAKEN: Refund of ${amount:.2f} escalated for manager review. Not processed automatically."


### Create the Agents

Same `create_agent` pattern as the other two notebooks, same local Ollama model, only the tool sets differ: naive_search_agent gets `naive_search`, semantic_search_agent gets `semantic_search_with_filter`, both get the same two action tools and the same instructions.

In [77]:
from langchain_ollama import ChatOllama
from langchain.agents import create_agent

llm = ChatOllama(
    model="gpt-oss:20b",
    temperature=0,  # 0 for deterministic decisions
    base_url="http://localhost:11434"  # Default Ollama URL
)

prompt = (
    "You are a CloudDeploy support agent handling refund requests. Use the search "
    "tool to check the current refund policy, then either approve the refund "
    "yourself or escalate it for manager review, based on what the policy says. "
    "Always take one of those two actions using the matching tool."
)

naive_search_agent = create_agent(
    model=llm,
    tools=[naive_search, approve_refund, escalate_for_review],
    system_prompt=prompt,
)

semantic_search_agent = create_agent(
    model=llm,
    tools=[semantic_search_with_filter, approve_refund, escalate_for_review],
    system_prompt=prompt,
)


### Ask Both Agents to Handle the Same Refund Request

Same task, same model, same instructions. The only variable is which search tool each agent has access to.

In [78]:
def print_response(messages):
    print(messages['messages'][-1].content)

task = "A customer is requesting a $75 refund. Check policy and take the appropriate action."

print("="*80)
print(" "*20 + "$75 REFUND REQUEST: NAIVE vs SEMANTIC WITH FILTERS RETRIEVAL")
print("="*80)

print("\n" + "─"*33 + " NAIVE RETRIEVAL " + "─"*31)
naive_response = naive_search_agent.invoke({"messages": [{"role": "user", "content": task}]})
print_response(naive_response)

print("\n" + "─"*33 + " SEMANTIC RETRIEVAL " + "─"*29)
semantic_search_response = semantic_search_agent.invoke({"messages": [{"role": "user", "content": task}]})
print_response(semantic_search_response)
print("="*80)


                    $75 REFUND REQUEST: NAIVE vs SEMANTIC WITH FILTERS RETRIEVAL

───────────────────────────────── NAIVE RETRIEVAL ───────────────────────────────
Refund of **$75** has been approved and processed automatically per the refund policy. No further action is required.

───────────────────────────────── SEMANTIC RETRIEVAL ─────────────────────────────
The refund request has been escalated for manager review as per the policy.


#### What to look for:
With naive retrieval, the agent should read the superseded 100-dollar threshold, conclude a 75-dollar refund qualifies for auto-approval, and call `approve_refund`, an actual refund that current policy says should have gone to a manager. With better (semantic with filter, for example) retrieval, the agent should read the current 50-dollar threshold, conclude the same 75-dollar refund needs review, and call `escalate_for_review` instead.

If both agents land on the same tool call, print `naive_search_result` and `semantic_search_result` above and check which document each one actually retrieved, that's the fastest way to see which link in the chain diverged.

### Why This Matters

A wrong *answer* is something a person reads, and might catch. A wrong *action* already happened by the time anyone reads anything. `approve_refund` above isn't a print statement standing in for nothing, it's standing in for `POST /refunds`, `grant_role()`, `send_wire_transfer()`, any real endpoint an agent might call once it's wired into MCP tools. The retrieval step decided the outcome. The agent's reasoning, tool-calling, and instruction-following were all working exactly as designed, the policy it reasoned over was just wrong.

### Summary

|  | Naive Retrieval | Better (Semantic with filter) Retrieval |
|-----------|----------|------------|
| **What it retrieves** | Whichever policy doc scores highest on literal overlap, ties broken arbitrarily | The current policy doc, explicitly filtered on freshness metadata |
| **What the agent does** | Approves a refund that should have been escalated | Escalates correctly |
| **What looks the same either way** | Tool-calling, reasoning, confidence, formatting of the final message | Same |

Agents don't fail by hesitating. They fail by acting, fluently, on whatever they were handed. As agents get more tools through MCP and more license to act rather than just answer, the retrieval step stops being a research nicety and becomes the thing standing between a working agent and one that's already done the wrong thing.